# Huấn luyện LTX 2.3 LoRA với Colab Notebook

Notebook này hướng dẫn huấn luyện LoRA cho mô hình LTX 2.3 sử dụng thư viện `musubi-tuner`.
Phiên bản này được tích hợp giao diện (Form) giúp dễ dàng tùy chỉnh thông số và hỗ trợ Mount Google Drive để lưu trữ Model/Dataset.

**Lưu ý**: Khuyến nghị GPU có 24GB VRAM trở lên (chọn GPU A100 hoặc L4/T4 trên Colab).

## Bước 1: Mount Google Drive & Khởi tạo Môi trường

Bước này sẽ gắn kết Google Drive của bạn để lưu/tải Model và Dataset, giúp bạn không cần tải lại ở các lần sau.

In [ ]:
#@title Khởi tạo môi trường & Mount Drive
mount_drive = True #@param {type:"boolean"}
drive_path = "/content/drive/MyDrive/LTX23_Workspace" #@param {type:"string"}

import os

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(drive_path, exist_ok=True)
    print(f"Đã mount Google Drive. Thư mục làm việc: {drive_path}")
    WORKSPACE = drive_path
else:
    WORKSPACE = "/content/workspace"
    os.makedirs(WORKSPACE, exist_ok=True)
    print(f"Sử dụng bộ nhớ tạm Colab. Thư mục làm việc: {WORKSPACE}")

# Clone Repo
repo_dir = os.path.join(WORKSPACE, "musubi-tuner")
if not os.path.exists(repo_dir):
    !git clone https://github.com/pmhaidn/musubi-tuner.git {repo_dir}

os.chdir(repo_dir)

# Cài đặt dependencies (chỉ chạy cài đặt trong môi trường Colab hiện tại)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -e .
!pip install ascii-magic matplotlib tensorboard accelerate huggingface_hub

## Bước 2: Tải Mô hình LTX 2.3 & Gemma

Mô hình sẽ được tải xuống thư mục làm việc (trên Drive nếu bạn đã Mount). Nếu mô hình đã tồn tại, bước này sẽ tự động bỏ qua để tiết kiệm thời gian.

In [ ]:
#@title Tải Model
import os
from huggingface_hub import hf_hub_download

model_dir = os.path.join(WORKSPACE, "models")
os.makedirs(model_dir, exist_ok=True)

ltx_path = os.path.join(model_dir, "ltx-2.3-22b-dev.safetensors")
gemma_path = os.path.join(model_dir, "text_encoders/gemma_3_12B_it_fp8_e4m3fn.safetensors")

if not os.path.exists(ltx_path):
    print("Đang tải LTX 2.3 Checkpoint...")
    ltx_path = hf_hub_download(repo_id="Lightricks/LTX-2.3", filename="ltx-2.3-22b-dev.safetensors", local_dir=model_dir)
else:
    print("LTX 2.3 Checkpoint đã tồn tại.")

if not os.path.exists(gemma_path):
    print("Đang tải Gemma FP8 Text Encoder...")
    gemma_path = hf_hub_download(repo_id="Kijai/LTX2.3_comfy", filename="text_encoders/gemma_3_12B_it_fp8_e4m3fn.safetensors", local_dir=model_dir)
else:
    print("Gemma FP8 Text Encoder đã tồn tại.")

print("Models sẵn sàng tại:", model_dir)

## Bước 3: Cấu hình Dataset và TOML

Sử dụng biểu mẫu bên dưới để chỉ định thư mục chứa video/ảnh và caption của bạn (đã tải lên Drive hoặc thông qua giao diện Upload của Colab). Code sẽ tự động sinh cấu hình `dataset.toml`.

In [ ]:
#@title Cấu hình Dataset
dataset_directory = "/content/drive/MyDrive/LTX23_Workspace/dataset/videos" #@param {type:"string"}
resolution_width = 768 #@param {type:"integer"}
resolution_height = 512 #@param {type:"integer"}
target_fps = 25 #@param {type:"integer"}
batch_size = 1 #@param {type:"integer"}

import os

os.makedirs(dataset_directory, exist_ok=True)
cache_directory = os.path.join(WORKSPACE, "dataset_cache")
os.makedirs(cache_directory, exist_ok=True)

toml_content = f"""[general]
resolution = [{resolution_width}, {resolution_height}]
caption_extension = ".txt"
batch_size = {batch_size}
enable_bucket = true

[[datasets]]
video_directory = "{dataset_directory}"
cache_directory = "{cache_directory}"
target_frames = [1, 17, 33]
target_fps = {target_fps}
"""

toml_path = os.path.join(repo_dir, "dataset.toml")
with open(toml_path, "w") as f:
    f.write(toml_content)

print(f"Đã tạo {toml_path} thành công.")
print(f"Hãy chắc chắn bạn đã upload dataset (video + .txt) vào: {dataset_directory}")

## Bước 4: Pre-caching

Mã hóa trước latents và text embeddings để tăng tốc độ huấn luyện. Việc cache sẽ được lưu vào Drive.

In [ ]:
#@title Cache Latents & Text Encoder Outputs
vae_chunk_size = 16 #@param {type:"integer"}

import subprocess

print("1. Caching Latents...")
subprocess.run([
    "python", "ltx2_cache_latents.py",
    "--dataset_config", "dataset.toml",
    "--ltx2_checkpoint", ltx_path,
    "--device", "cuda",
    "--vae_dtype", "bf16",
    "--vae_chunk_size", str(vae_chunk_size),
    "--ltx2_mode", "video"
], check=True)

print("2. Caching Text Encoder Outputs...")
subprocess.run([
    "python", "ltx2_cache_text_encoder_outputs.py",
    "--dataset_config", "dataset.toml",
    "--ltx2_checkpoint", ltx_path,
    "--gemma_safetensors", gemma_path,
    "--device", "cuda",
    "--mixed_precision", "bf16",
    "--ltx2_mode", "video",
    "--batch_size", "1"
], check=True)
print("Hoàn tất Pre-caching!")

## Bước 5: Huấn luyện LoRA

Sử dụng giao diện để tinh chỉnh tham số huấn luyện (learning rate, số epochs, v.v.) và nhấn chạy để bắt đầu quá trình.

In [ ]:
#@title Tham số Huấn luyện
learning_rate = "1e-4" #@param {type:"string"}
max_train_epochs = 10 #@param {type:"integer"}
save_every_n_epochs = 5 #@param {type:"integer"}
network_dim = 32 #@param {type:"integer"}
network_alpha = 32 #@param {type:"integer"}
blocks_to_swap = 30 #@param {type:"integer"}
output_name = "ltx23_lora" #@param {type:"string"}

output_dir = os.path.join(WORKSPACE, "output")
os.makedirs(output_dir, exist_ok=True)

train_cmd = [
    "accelerate", "launch", "--num_cpu_threads_per_process", "1", "ltx2_train_network.py",
    "--mixed_precision", "bf16",
    "--dataset_config", "dataset.toml",
    "--ltx2_checkpoint", ltx_path,
    "--ltx_version", "2.3",
    "--ltx_version_check_mode", "warn",
    "--ltx2_mode", "video",
    "--fp8_base", "--fp8_scaled",
    "--blocks_to_swap", str(blocks_to_swap),
    "--use_pinned_memory_for_block_swap",
    "--gradient_checkpointing",
    "--gradient_checkpointing_cpu_offload",
    "--sdpa",
    "--learning_rate", learning_rate,
    "--network_module", "networks.lora_ltx2",
    "--network_dim", str(network_dim),
    "--network_alpha", str(network_alpha),
    "--timestep_sampling", "shifted_logit_normal",
    "--output_dir", output_dir,
    "--output_name", output_name,
    "--max_train_epochs", str(max_train_epochs),
    "--save_every_n_epochs", str(save_every_n_epochs)
]

import subprocess
subprocess.run(train_cmd, check=True)
print(f"Huấn luyện hoàn tất. LoRA được lưu tại: {output_dir}")